# PHM08 (PHM 2008 Challenge) — Federated Learning Pipeline (CMAPSS-style)

This notebook is the **FL version** (not centralized). It follows the same CMAPSS-style pipeline:
- Sliding windows per engine
- Dual-head DS-TCN: **fault probability** + **RUL quantiles**
- Physics-guided monotonic constraint (q50 non-increasing)
- **Federated training (FedAvg)** over clients (airlines/MROs) created by splitting engine units
- Validation on held-out engines from `train.txt` (PHM08 test labels are hidden)
- Saves: round curves (PDF), global + per-client metrics (JSON/CSV) in `results/phm08_fl/`

**Expected files**:
- `dataset/train.txt`, `dataset/test.txt`, `dataset/final_test.txt`

> Note: Metrics like MAE/AUPRC are computed on **validation from train** (same reason as before).


In [ ]:

# ============ Imports ============
import os, json, math, random, time, copy
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, roc_curve, precision_recall_curve
import matplotlib.pyplot as plt

def seed_all(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_all(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

ROOT = Path.cwd()
DATA_DIR = ROOT / "dataset"
RES_DIR  = ROOT / "results" / "phm08_fl"
(RES_DIR / "plots").mkdir(parents=True, exist_ok=True)
(RES_DIR / "metrics").mkdir(parents=True, exist_ok=True)
(RES_DIR / "preds").mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 300,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "legend.fontsize": 10,
    "lines.linewidth": 2.0,
})




DEVICE: cuda


## 1) Load PHM08 + compute train RUL


In [2]:

PHM08_COLS = ["unit","cycle","op1","op2","op3"] + [f"s{i}" for i in range(1,22)]
FEAT_COLS  = ["op1","op2","op3"] + [f"s{i}" for i in range(1,22)]

def read_phm08_txt(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep=r"\s+", header=None)
    df = df.iloc[:, :26]
    df.columns = PHM08_COLS
    df["unit"] = df["unit"].astype(int)
    df["cycle"] = df["cycle"].astype(int)
    return df

def compute_rul_train(df_train: pd.DataFrame, rul_cap: Optional[int] = None) -> pd.DataFrame:
    df = df_train.copy()
    max_cycle = df.groupby("unit")["cycle"].max().to_dict()
    df["RUL"] = df.apply(lambda r: float(max_cycle[int(r["unit"])] - int(r["cycle"])), axis=1)
    if rul_cap is not None:
        df["RUL"] = np.minimum(df["RUL"].values, float(rul_cap))
    return df

df_train_raw = read_phm08_txt(DATA_DIR/"train.txt")
df_test_raw  = read_phm08_txt(DATA_DIR/"test.txt")
df_final_raw = read_phm08_txt(DATA_DIR/"final_test.txt")

df_train = compute_rul_train(df_train_raw, rul_cap=None)
print("train:", df_train.shape, "units:", df_train["unit"].nunique())
print("test :", df_test_raw.shape, "units:", df_test_raw["unit"].nunique())
print("final:", df_final_raw.shape, "units:", df_final_raw["unit"].nunique())
df_train.head()


train: (45918, 27) units: 218
test : (29820, 26) units: 218
final: (55156, 26) units: 435


,unit,cycle,op1,op2,op3,s1,s2,s3,s4,s5,...,s13,s14,s15,s16,s17,s18,s19,s20,s21,RUL
0,1,1,10.0047,0.2501,20.0,489.05,604.13,1499.45,1309.95,10.52,...,2388.13,8120.83,8.6216,0.03,368,2319,100.0,28.58,17.1735,222.0
1,1,2,0.0015,0.0003,100.0,518.67,642.13,1584.55,1403.96,14.62,...,2388.15,8132.87,8.3907,0.03,391,2388,100.0,38.99,23.3619,221.0
2,1,3,34.9986,0.8401,60.0,449.44,555.42,1368.17,1122.49,5.48,...,2387.95,8063.84,9.3557,0.02,334,2223,100.0,14.83,8.8555,220.0
3,1,4,20.0031,0.7005,0.0,491.19,607.03,1488.44,1249.18,9.35,...,2388.07,8052.30,9.2231,0.02,364,2324,100.0,24.42,14.7832,219.0
4,1,5,42.0041,0.8405,40.0,445.00,549.52,1354.48,1124.32,3.91,...,2387.89,8083.67,9.2986,0.02,330,2212,100.0,10.99,6.4025,218.0


## 2) Windowing + datasets + standardization


In [3]:

@dataclass
class WindowConfig:
    W: int = 30
    stride: int = 1
    tau: int = 30
    rul_cap: Optional[int] = None

def make_windows_for_units(df: pd.DataFrame, unit_ids: List[int], cfg: WindowConfig) -> Dict[str, np.ndarray]:
    W, stride = cfg.W, cfg.stride
    X_list, y_rul_list, y_fault_list, unit_list, endc_list = [], [], [], [], []
    for u in unit_ids:
        d = df[df["unit"] == u].sort_values("cycle").reset_index(drop=True)
        feats = d[FEAT_COLS].values.astype(np.float32)
        rul = d["RUL"].values.astype(np.float32)
        cycles = d["cycle"].values.astype(np.int32)

        n = len(d)
        if n < W:
            continue

        for i in range(0, n - W + 1, stride):
            x = feats[i:i+W]
            rul_end = float(rul[i+W-1])
            fault = 1.0 if rul_end <= cfg.tau else 0.0
            X_list.append(x)
            y_rul_list.append(rul_end)
            y_fault_list.append(fault)
            unit_list.append(int(u))
            endc_list.append(int(cycles[i+W-1]))

    return {
        "X": np.array(X_list, dtype=np.float32),
        "y_rul": np.array(y_rul_list, dtype=np.float32),
        "y_fault": np.array(y_fault_list, dtype=np.float32),
        "unit": np.array(unit_list, dtype=np.int32),
        "end_cycle": np.array(endc_list, dtype=np.int32),
    }

class WindowDataset(Dataset):
    def __init__(self, arrays: Dict[str, np.ndarray]):
        self.X = arrays["X"]; self.y_rul = arrays["y_rul"]; self.y_fault = arrays["y_fault"]
        self.unit = arrays["unit"]; self.end_cycle = arrays["end_cycle"]
    def __len__(self): return len(self.X)
    def __getitem__(self, idx):
        return (torch.from_numpy(self.X[idx]),
                torch.tensor(self.y_rul[idx]),
                torch.tensor(self.y_fault[idx]),
                torch.tensor(self.unit[idx]),
                torch.tensor(self.end_cycle[idx]))

def fit_standardizer(X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    mu = X.reshape(-1, X.shape[-1]).mean(axis=0)
    sd = X.reshape(-1, X.shape[-1]).std(axis=0) + 1e-6
    return mu.astype(np.float32), sd.astype(np.float32)

def apply_standardizer(X: np.ndarray, mu: np.ndarray, sd: np.ndarray) -> np.ndarray:
    return ((X - mu[None,None,:]) / sd[None,None,:]).astype(np.float32)


## 3) Create FL clients from engines + held-out validation engines
We simulate airlines/MROs as clients by splitting engine units into `K` disjoint groups.


In [4]:

@dataclass
class SplitConfig:
    val_ratio: float = 0.2
    n_clients: int = 8
    seed: int = 42
    shard_by_rul_length: bool = True  # non-IID: sort by life length and shard

def split_units_for_fl(df_train: pd.DataFrame, cfg: SplitConfig) -> Tuple[List[List[int]], List[int]]:
    units = sorted(df_train["unit"].unique().tolist())
    rng = np.random.default_rng(cfg.seed)

    # Engine life length for non-IID sharding
    life = df_train.groupby("unit")["cycle"].max().to_dict()
    if cfg.shard_by_rul_length:
        units = sorted(units, key=lambda u: life[u])

    # Hold out some engines for validation
    n_val = max(1, int(len(units) * cfg.val_ratio))
    val_units = units[-n_val:]
    train_units = units[:-n_val]

    # Split remaining into clients
    if cfg.shard_by_rul_length:
        # contiguous shards -> stronger non-IID
        shards = np.array_split(np.array(train_units, dtype=int), cfg.n_clients)
        clients = [sorted(s.tolist()) for s in shards]
    else:
        rng.shuffle(train_units)
        shards = np.array_split(np.array(train_units, dtype=int), cfg.n_clients)
        clients = [sorted(s.tolist()) for s in shards]

    return clients, sorted(val_units)

scfg = SplitConfig(val_ratio=0.2, n_clients=8, seed=42, shard_by_rul_length=True)
clients_units, val_units = split_units_for_fl(df_train, scfg)
print("Clients:", len(clients_units), "| Val units:", len(val_units))
print("Client sizes:", [len(c) for c in clients_units])


Clients: 8 | Val units: 43
Client sizes: [22, 22, 22, 22, 22, 22, 22, 21]


## 4) Model + losses (same as CMAPSS pipeline)


In [5]:

class DepthwiseSeparableTCNBlock(nn.Module):
    def __init__(self, channels: int, out_channels: int, kernel_size: int, dilation: int, dropout: float):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2
        self.dw = nn.Conv1d(channels, channels, kernel_size=kernel_size, dilation=dilation,
                            padding=padding, groups=channels, bias=False)
        self.pw = nn.Conv1d(channels, out_channels, kernel_size=1, bias=True)
        self.act = nn.ReLU()
        self.drop = nn.Dropout(dropout)
        self.use_res = (channels == out_channels)
    def forward(self, x):
        y = self.dw(x)
        y = self.pw(y)
        y = self.act(y)
        y = self.drop(y)
        if self.use_res:
            y = y + x
        return y

class DS_TCN_Encoder(nn.Module):
    def __init__(self, in_channels: int, hidden: int = 64, kernel_size: int = 3, dilations=(1,2,4,8), dropout: float = 0.1):
        super().__init__()
        layers = []
        ch = in_channels
        for d in dilations:
            layers.append(DepthwiseSeparableTCNBlock(ch, hidden, kernel_size=kernel_size, dilation=d, dropout=dropout))
            ch = hidden
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

class DualHeadModel(nn.Module):
    def __init__(self, n_features: int, hidden: int = 64, dropout: float = 0.1):
        super().__init__()
        self.enc = DS_TCN_Encoder(in_channels=n_features, hidden=hidden, dropout=dropout)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fault_head = nn.Sequential(
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, 1)
        )
        self.rul_head = nn.Sequential(
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, 3)
        )
    def forward(self, x):
        x = x.permute(0,2,1)
        z = self.enc(x)
        h = self.pool(z).squeeze(-1)
        p_fault = torch.sigmoid(self.fault_head(h)).squeeze(-1)
        q = self.rul_head(h)
        q10, q50, q90 = q[:,0], q[:,1], q[:,2]
        q50 = torch.maximum(q50, q10)
        q90 = torch.maximum(q90, q50)
        q = torch.stack([q10,q50,q90], dim=1)
        return {"p_fault": p_fault, "q": q, "z": z, "h": h}

ALPHAS = (0.1, 0.5, 0.9)

def pinball_loss(q: torch.Tensor, y: torch.Tensor, alphas=ALPHAS) -> torch.Tensor:
    losses = []
    for i, a in enumerate(alphas):
        e = y - q[:, i]
        losses.append(torch.maximum(a*e, (a-1)*e).mean())
    return sum(losses)

def focal_loss(p: torch.Tensor, y: torch.Tensor, alpha: float, gamma: float = 2.0, eps: float = 1e-6) -> torch.Tensor:
    p = torch.clamp(p, eps, 1-eps)
    pt = torch.where(y == 1, p, 1-p)
    w = torch.where(y == 1, torch.tensor(alpha, device=p.device), torch.tensor(1-alpha, device=p.device))
    return (-w * (1-pt)**gamma * torch.log(pt)).mean()

def monotonic_loss(q50_t: torch.Tensor, q50_t1: torch.Tensor) -> torch.Tensor:
    return F.relu(q50_t1 - q50_t).mean()

def batch_alpha_from_imbalance(y_fault: torch.Tensor) -> float:
    pos = float((y_fault > 0.5).float().mean().item())
    return float(np.clip(0.5 / max(pos, 1e-4), 0.5, 0.99))


## 5) Metrics + saving helpers


In [6]:

def compute_fault_metrics(y_true: np.ndarray, y_prob: np.ndarray) -> Dict[str, float]:
    if len(y_true) == 0 or len(np.unique(y_true)) < 2:
        return {"auprc": float("nan"), "auroc": float("nan"), "f1@0.5": float("nan"), "recall@fpr1%": float("nan")}
    out = {}
    out["auprc"] = float(average_precision_score(y_true, y_prob))
    out["auroc"] = float(roc_auc_score(y_true, y_prob))
    out["f1@0.5"] = float(f1_score(y_true, (y_prob >= 0.5).astype(int)))
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    mask = fpr <= 0.01
    out["recall@fpr1%"] = float(np.max(tpr[mask])) if np.any(mask) else 0.0
    return out

def compute_rul_metrics(y_true: np.ndarray, q: np.ndarray) -> Dict[str, float]:
    q50 = q[:, 1]
    mae = float(np.mean(np.abs(q50 - y_true)))
    rmse = float(np.sqrt(np.mean((q50 - y_true)**2)))
    cov = float(np.mean((y_true >= q[:,0]) & (y_true <= q[:,2])))
    iw  = float(np.mean(q[:,2] - q[:,0]))
    return {"mae": mae, "rmse": rmse, "coverage_q10_q90": cov, "mean_interval_width": iw}

def save_json(obj: Dict, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def plot_round_curves(hist: Dict[str, List[float]], outpath: Path, title: str, ylabel: str):
    plt.figure()
    plt.plot(hist["round"], hist["value"])
    plt.xlabel("round"); plt.ylabel(ylabel); plt.title(title)
    plt.tight_layout(); plt.savefig(outpath); plt.close()

def plot_loss_curve(train_loss: List[float], outpath: Path):
    plt.figure()
    plt.plot(train_loss)
    plt.xlabel("round"); plt.ylabel("train loss"); plt.title("FL training loss per round")
    plt.tight_layout(); plt.savefig(outpath); plt.close()


## 6) FL training (FedAvg) — same pipeline as CMAPSS
You can adjust:
- `n_rounds`
- `clients_per_round`
- `local_epochs`
- trimming / robust aggregation (optional later)


In [7]:

@dataclass
class FLConfig:
    W: int = 30
    stride: int = 1
    tau: int = 30
    rul_cap: Optional[int] = None

    n_clients: int = 8
    clients_per_round: int = 4
    n_rounds: int = 50
    local_epochs: int = 1
    batch_size: int = 128

    lr: float = 1e-3
    weight_decay: float = 1e-4
    hidden: int = 64
    dropout: float = 0.1

    lambda_rul: float = 1.0
    lambda_fault: float = 1.0
    lambda_mono: float = 0.1
    focal_gamma: float = 2.0

    seed: int = 42

def build_pair_map(arrays: Dict[str, np.ndarray]) -> Dict[int, Dict[int, int]]:
    pm = {}
    for idx, (u, c) in enumerate(zip(arrays["unit"], arrays["end_cycle"])):
        pm.setdefault(int(u), {})[int(c)] = int(idx)
    return pm

def sample_pairs(pm: Dict[int, Dict[int,int]], units: np.ndarray, endc: np.ndarray, max_pairs: int):
    pairs = []
    for u, c in zip(units, endc):
        u = int(u); c = int(c)
        nxt = c + 1
        if u in pm and nxt in pm[u]:
            pairs.append((pm[u][c], pm[u][nxt]))
        if len(pairs) >= max_pairs:
            break
    return pairs

def get_model_weights(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

def set_model_weights(model: nn.Module, weights: Dict[str, torch.Tensor]):
    state = model.state_dict()
    for k in state.keys():
        state[k] = weights[k].to(state[k].device)
    model.load_state_dict(state)

def fedavg(weight_list: List[Dict[str, torch.Tensor]], n_list: List[int]) -> Dict[str, torch.Tensor]:
    total = float(sum(n_list))
    out = {}
    for k in weight_list[0].keys():
        out[k] = sum(w[k] * (n/total) for w, n in zip(weight_list, n_list))
    return out

def evaluate_model(model: nn.Module, loader: DataLoader) -> Dict[str, Dict[str, float]]:
    model.eval()
    y_true_fault, y_prob_fault, y_true_rul, y_q = [], [], [], []
    with torch.no_grad():
        for x, y_rul, y_fault, unit, endc in loader:
            out = model(x.to(DEVICE))
            y_true_fault.append(y_fault.numpy().astype(int))
            y_prob_fault.append(out["p_fault"].detach().cpu().numpy())
            y_true_rul.append(y_rul.numpy())
            y_q.append(out["q"].detach().cpu().numpy())
    y_true_fault = np.concatenate(y_true_fault) if y_true_fault else np.array([])
    y_prob_fault = np.concatenate(y_prob_fault) if y_prob_fault else np.array([])
    y_true_rul   = np.concatenate(y_true_rul) if y_true_rul else np.array([])
    y_q          = np.concatenate(y_q) if y_q else np.array([])
    return {"fault": compute_fault_metrics(y_true_fault, y_prob_fault),
            "rul": compute_rul_metrics(y_true_rul, y_q)}

def train_one_client(global_weights: Dict[str, torch.Tensor],
                     client_arrays: Dict[str, np.ndarray],
                     mu: np.ndarray, sd: np.ndarray,
                     cfg: FLConfig) -> Tuple[Dict[str, torch.Tensor], int, float]:
    # Standardize
    arrays = {k: v for k, v in client_arrays.items()}
    arrays["X"] = apply_standardizer(arrays["X"], mu, sd)

    ds = WindowDataset(arrays)
    loader = DataLoader(ds, batch_size=cfg.batch_size, shuffle=True, drop_last=True)

    model = DualHeadModel(n_features=len(FEAT_COLS), hidden=cfg.hidden, dropout=cfg.dropout).to(DEVICE)
    set_model_weights(model, global_weights)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    pm = build_pair_map(arrays)
    pair_idx = sample_pairs(pm, arrays["unit"], arrays["end_cycle"], max_pairs=10000)

    model.train()
    losses = []
    for _ in range(cfg.local_epochs):
        for x, y_rul, y_fault, unit, endc in loader:
            x = x.to(DEVICE); y_rul = y_rul.to(DEVICE); y_fault = y_fault.to(DEVICE)
            out = model(x)
            p = out["p_fault"]; q = out["q"]

            alpha = batch_alpha_from_imbalance(y_fault)
            lf = focal_loss(p, y_fault, alpha=alpha, gamma=cfg.focal_gamma)
            lrul = pinball_loss(q, y_rul)

            lmono = torch.tensor(0.0, device=DEVICE)
            if len(pair_idx) > 0:
                k = min(1024, len(pair_idx))
                sel = np.random.choice(len(pair_idx), size=k, replace=False)
                i0 = torch.tensor([pair_idx[s][0] for s in sel], device=DEVICE)
                i1 = torch.tensor([pair_idx[s][1] for s in sel], device=DEVICE)
                x0 = torch.from_numpy(arrays["X"][i0.cpu().numpy()]).to(DEVICE)
                x1 = torch.from_numpy(arrays["X"][i1.cpu().numpy()]).to(DEVICE)
                q0 = model(x0)["q"][:,1]
                q1 = model(x1)["q"][:,1]
                lmono = monotonic_loss(q0, q1)

            loss = cfg.lambda_fault*lf + cfg.lambda_rul*lrul + cfg.lambda_mono*lmono

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            losses.append(float(loss.item()))

    return get_model_weights(model), len(ds), float(np.mean(losses)) if losses else float("nan")

def run_federated(cfg: FLConfig):
    seed_all(cfg.seed)

    # Build windows once per client (unstandardized); standardize using global TRAIN (all client-train windows)
    wcfg = WindowConfig(W=cfg.W, stride=cfg.stride, tau=cfg.tau, rul_cap=cfg.rul_cap)

    # Prepare validation arrays
    val_arrays = make_windows_for_units(df_train, val_units, wcfg)

    # Prepare client arrays
    client_arrays_list = []
    for cid, units in enumerate(clients_units):
        arr = make_windows_for_units(df_train, units, wcfg)
        client_arrays_list.append(arr)

    # Fit global standardizer on all client-train data pooled
    X_pool = np.concatenate([c["X"] for c in client_arrays_list if len(c["X"]) > 0], axis=0)
    mu, sd = fit_standardizer(X_pool)
    np.save(RES_DIR/"metrics"/"standardizer_mu.npy", mu)
    np.save(RES_DIR/"metrics"/"standardizer_sd.npy", sd)

    # Standardize val
    val_arrays["X"] = apply_standardizer(val_arrays["X"], mu, sd)
    val_loader = DataLoader(WindowDataset(val_arrays), batch_size=cfg.batch_size, shuffle=False)

    # Init global model
    global_model = DualHeadModel(n_features=len(FEAT_COLS), hidden=cfg.hidden, dropout=cfg.dropout).to(DEVICE)
    global_weights = get_model_weights(global_model)

    history = {"round": [], "train_loss": [], "val_mae": [], "val_auprc": []}
    best_mae = float("inf")
    best_ckpt = RES_DIR/"metrics"/"best_global.pt"

    for r in range(1, cfg.n_rounds+1):
        t0 = time.time()
        # sample clients
        rng = np.random.default_rng(cfg.seed + r)
        sel = sorted(rng.choice(len(client_arrays_list), size=min(cfg.clients_per_round, len(client_arrays_list)), replace=False).tolist())

        weight_updates, n_list, loss_list = [], [], []
        for cid in sel:
            w, n, l = train_one_client(global_weights, client_arrays_list[cid], mu, sd, cfg)
            weight_updates.append(w); n_list.append(n); loss_list.append(l)

        # aggregate
        global_weights = fedavg(weight_updates, n_list)
        set_model_weights(global_model, global_weights)

        # eval global on val
        val_metrics = evaluate_model(global_model, val_loader)
        val_mae = val_metrics["rul"]["mae"]
        val_auprc = val_metrics["fault"]["auprc"]
        round_loss = float(np.nanmean(loss_list))

        history["round"].append(r)
        history["train_loss"].append(round_loss)
        history["val_mae"].append(val_mae)
        history["val_auprc"].append(val_auprc)

        # save best by MAE (you can change to composite)
        if val_mae < best_mae:
            best_mae = val_mae
            torch.save({
                "global_state": global_model.state_dict(),
                "cfg": asdict(cfg),
                "mu": mu, "sd": sd,
                "feat_cols": FEAT_COLS,
                "val_units": val_units,
                "clients_units": clients_units,
                "best_round": r,
                "best_val_metrics": val_metrics,
            }, best_ckpt)

        print(f"Round {r:03d}/{cfg.n_rounds} | sel={sel} | train_loss={round_loss:.4f} | val_mae={val_mae:.3f} | val_auprc={val_auprc:.3f} | best_mae={best_mae:.3f} | {time.time()-t0:.1f}s")

    # Save curves
    plot_loss_curve(history["train_loss"], RES_DIR/"plots"/"fl_train_loss_round.pdf")
    plot_round_curves({"round": history["round"], "value": history["val_mae"]}, RES_DIR/"plots"/"fl_val_mae_round.pdf",
                      "Validation MAE vs FL round", "MAE")
    plot_round_curves({"round": history["round"], "value": history["val_auprc"]}, RES_DIR/"plots"/"fl_val_auprc_round.pdf",
                      "Validation AUPRC vs FL round", "AUPRC")

    save_json({"history": history, "best_mae": best_mae}, RES_DIR/"metrics"/"fl_history.json")
    return best_ckpt

flcfg = FLConfig(n_clients=scfg.n_clients, n_rounds=50, clients_per_round=4, local_epochs=1, W=30, tau=30, batch_size=128)
best_ckpt = run_federated(flcfg)
print("Best checkpoint:", best_ckpt)


Round 001/50 | sel=[0, 2, 3, 6] | train_loss=114.3706 | val_mae=119.866 | val_auprc=0.126 | best_mae=119.866 | 3.8s
Round 002/50 | sel=[0, 2, 3, 5] | train_loss=92.2430 | val_mae=84.170 | val_auprc=0.125 | best_mae=84.170 | 2.2s
Round 003/50 | sel=[3, 4, 5, 7] | train_loss=55.4394 | val_mae=62.690 | val_auprc=0.118 | best_mae=62.690 | 2.3s
Round 004/50 | sel=[0, 1, 2, 5] | train_loss=38.7994 | val_mae=63.509 | val_auprc=0.110 | best_mae=62.690 | 1.9s
Round 005/50 | sel=[0, 3, 4, 6] | train_loss=42.8288 | val_mae=62.448 | val_auprc=0.101 | best_mae=62.448 | 2.1s
Round 006/50 | sel=[0, 2, 3, 4] | train_loss=38.9993 | val_mae=63.411 | val_auprc=0.093 | best_mae=62.448 | 1.9s
Round 007/50 | sel=[0, 2, 4, 5] | train_loss=40.2501 | val_mae=62.836 | val_auprc=0.085 | best_mae=62.448 | 2.0s
Round 008/50 | sel=[3, 4, 5, 6] | train_loss=45.8740 | val_mae=60.617 | val_auprc=0.080 | best_mae=60.617 | 2.4s
Round 009/50 | sel=[1, 2, 5, 6] | train_loss=42.0390 | val_mae=60.148 | val_auprc=0.076 | bes

## 7) Final evaluation on validation + per-client fairness report


In [8]:

# Load best global model
ckpt = torch.load(best_ckpt, map_location=DEVICE)
mu, sd = ckpt["mu"], ckpt["sd"]
cfg = ckpt["cfg"]

best_model = DualHeadModel(n_features=len(FEAT_COLS), hidden=cfg["hidden"], dropout=cfg["dropout"]).to(DEVICE)
best_model.load_state_dict(ckpt["global_state"])
best_model.eval()

wcfg = WindowConfig(W=cfg["W"], stride=cfg["stride"], tau=cfg["tau"], rul_cap=cfg["rul_cap"])

# Validation metrics (global)
val_arrays = make_windows_for_units(df_train, ckpt["val_units"], wcfg)
val_arrays["X"] = apply_standardizer(val_arrays["X"], mu, sd)
val_loader = DataLoader(WindowDataset(val_arrays), batch_size=cfg["batch_size"], shuffle=False)
val_metrics = evaluate_model(best_model, val_loader)

# Per-client metrics (on each client's own data) for fairness diagnosis
client_reports = []
for cid, units in enumerate(ckpt["clients_units"]):
    arr = make_windows_for_units(df_train, units, wcfg)
    if len(arr["X"]) == 0:
        continue
    arr["X"] = apply_standardizer(arr["X"], mu, sd)
    loader = DataLoader(WindowDataset(arr), batch_size=cfg["batch_size"], shuffle=False)
    m = evaluate_model(best_model, loader)
    client_reports.append({
        "client_id": int(cid),
        "n_units": int(len(units)),
        "n_windows": int(len(arr["X"])),
        "fault": m["fault"],
        "rul": m["rul"],
    })

df_clients = pd.DataFrame([{
    "client_id": r["client_id"],
    "n_units": r["n_units"],
    "n_windows": r["n_windows"],
    "auprc": r["fault"]["auprc"],
    "recall@fpr1%": r["fault"]["recall@fpr1%"],
    "mae": r["rul"]["mae"],
    "rmse": r["rul"]["rmse"],
    "coverage": r["rul"]["coverage_q10_q90"],
} for r in client_reports]).sort_values("mae", ascending=False)

df_clients.to_csv(RES_DIR/"metrics"/"per_client_metrics.csv", index=False)

summary = {
    "val_metrics": val_metrics,
    "worst_client_by_mae": df_clients.iloc[0].to_dict() if len(df_clients) else None,
    "best_client_by_mae": df_clients.iloc[-1].to_dict() if len(df_clients) else None,
}
save_json(summary, RES_DIR/"metrics"/"final_eval_and_fairness.json")

print("Global validation metrics:", json.dumps(val_metrics, indent=2))
display(df_clients.head(10))


C:\Users\gharamik\AppData\Local\Temp\ipykernel_14072\1427230216.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(best_ckpt, map_location=DEVICE)


Global validation metrics: {
  "fault": {
    "auprc": 0.8628781602888987,
    "auroc": 0.9771554670493646,
    "f1@0.5": 0.2232269948924056,
    "recall@fpr1%": 0.6436609152288072
  },
  "rul": {
    "mae": 33.96818923950195,
    "rmse": 46.14917755126953,
    "coverage_q10_q90": 0.2530631479736098,
    "mean_interval_width": 46.028011322021484
  }
}


,client_id,n_units,n_windows,auprc,recall@fpr1%,mae,rmse,coverage
0,0,22,2557,0.898577,0.564516,46.400093,50.977444,0.410246
1,1,22,2921,0.902139,0.605572,40.326832,46.911900,0.521397
2,2,22,3256,0.887906,0.624633,31.871943,41.660988,0.522113
4,4,22,3837,0.910487,0.599707,25.089922,31.633512,0.579359
3,3,22,3581,0.921533,0.623167,24.338427,32.534107,0.576655
7,7,21,4374,0.907261,0.648233,23.162991,30.533873,0.379287
6,6,22,4345,0.914283,0.646628,22.353352,28.332140,0.458228
5,5,22,4115,0.969665,0.813783,21.532423,26.031553,0.666586


## 8) Inference on test/final_test (one prediction per unit)


In [9]:

def make_windows_inference(df_raw: pd.DataFrame, mu: np.ndarray, sd: np.ndarray, cfg: WindowConfig) -> Dict[str, np.ndarray]:
    df = df_raw.copy()
    unit_ids = sorted(df["unit"].unique().tolist())
    df["RUL"] = 0.0
    arrays = make_windows_for_units(df, unit_ids, cfg)
    arrays["X"] = apply_standardizer(arrays["X"], mu, sd)
    return arrays

def predict_last_per_unit(model: nn.Module, arrays: Dict[str, np.ndarray], batch_size: int = 256) -> pd.DataFrame:
    ds = WindowDataset(arrays)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    p_all, q_all, unit_all, endc_all = [], [], [], []
    with torch.no_grad():
        for x, y_rul, y_fault, unit, endc in loader:
            out = model(x.to(DEVICE))
            p_all.append(out["p_fault"].detach().cpu().numpy())
            q_all.append(out["q"].detach().cpu().numpy())
            unit_all.append(unit.numpy())
            endc_all.append(endc.numpy())
    p_all = np.concatenate(p_all); q_all = np.concatenate(q_all)
    unit_all = np.concatenate(unit_all); endc_all = np.concatenate(endc_all)

    rows = []
    for u in np.unique(unit_all):
        mask = (unit_all == u)
        idx = np.argmax(endc_all[mask])
        true_idx = np.where(mask)[0][idx]
        rows.append({
            "unit": int(u),
            "end_cycle": int(endc_all[true_idx]),
            "p_fault": float(p_all[true_idx]),
            "q10": float(q_all[true_idx,0]),
            "q50": float(q_all[true_idx,1]),
            "q90": float(q_all[true_idx,2]),
            "rul_hat": float(q_all[true_idx,1]),
        })
    return pd.DataFrame(rows).sort_values("unit").reset_index(drop=True)

test_arrays  = make_windows_inference(df_test_raw,  mu, sd, wcfg)
final_arrays = make_windows_inference(df_final_raw, mu, sd, wcfg)

df_pred_test  = predict_last_per_unit(best_model, test_arrays)
df_pred_final = predict_last_per_unit(best_model, final_arrays)

df_pred_test.to_csv(RES_DIR/"preds"/"test_preds_last_per_unit.csv", index=False)
df_pred_final.to_csv(RES_DIR/"preds"/"final_test_preds_last_per_unit.csv", index=False)

df_pred_test.head()


,unit,end_cycle,p_fault,q10,q50,q90,rul_hat
0,1,54,0.571498,75.819389,137.606491,137.606491,137.606491
1,2,157,0.576383,49.129684,89.576813,89.576813,89.576813
2,3,116,0.571498,65.787460,119.545250,119.545250,119.545250
3,4,74,0.571498,92.009438,166.751068,166.751068,166.751068
4,5,218,0.789325,25.158979,46.443073,46.443073,46.443073
